### nxsut generator

Run this notebook to generate or update the nxsut reference database from Exiobase Hybrid v3.3.18.

This version always produces three exports for the selected `year`:

* `v1.0`: legacy supply-mix update, matching the original notebook.
* `v2.0`: legacy supply-mix plus legacy electricity trade update, matching the original notebook.
* `v2.1`: new MARIO-native pipeline, exported separately under `v2.1/<year>`.

Set `user` and `year` in the first code cell, then run the notebook from top to bottom.

In [1]:
import mario
import yaml
import pandas as pd
import os

from support.ember_remapping import map_ember_to_classification
import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
year = 2024   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

---
## v1.0 and v2.0 - legacy pipeline

This section intentionally keeps the old logic for `v1.0` and `v2.0`: manual EMBER remapping, manual supply-market-share update, manual pooled electricity layer, and `shock_calc` with the legacy trade workbook. Only the deprecated pandas `groupby(axis=1)` syntax is written in the pandas >= 3 equivalent form.

Parse raw Exiobase database

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

Aggregating electricity commodities and activities to match EMBER resolution

In [ ]:
db.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

In [ ]:
# Parse ember electricity generation data, map to exiobase and get electricity mix for a given year 
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = None,
    mode = 'mix',
)

Updating electricity supply mixes (legacy manual loop)

In [ ]:
z = db.z
s = db.s

for region in db.get_index('Region'):
    print(region, end=' ')
    region_latest_year = ee_mix.loc[(region, slice(None), slice(None))].index.get_level_values(0).max()
    mix_year = year if year <= region_latest_year else region_latest_year
    
    new_mix = ee_mix.loc[(region, mix_year, slice(None)), 'Value'].to_frame().sort_index(axis=0)
    new_mix.index = new_mix.index.get_level_values(2)
    
    old_market_share = s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')].sum().sum()
    
    s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')] = new_mix['Value'].values * old_market_share
    print('done')

z.update(s)

db.update_scenarios('baseline', z=z)
db.reset_to_coefficients('baseline')

Export the v1.0 database

In [ ]:
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v1.0",
        str(year),
        ),
    )

---
## v2.0 - legacy electricity trades

Define the commodities whose trade mixes are updated and add the related legacy pass-through sectors as empty rows and columns.

In [ ]:
traded_commodities = ['Electricity']

for commodity in traded_commodities:
    new_activities = [f"{commodity} supply"]
    new_commdodities = [f"{commodity} need"]

db.add_sectors(
    new_sectors = new_activities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_activities.xlsx",
    item = "Activity",
    inplace = True,
)

db.add_sectors(
    new_sectors = new_commdodities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_commodities.xlsx",
    item = "Commodity",
    inplace = True,
)

### Demand-side shock

1. Activities supplying the new commodities consume only the domestic original commodity.
2. The consumption of the original commodity is transferred to domestic `need` commodity consumption, both for intermediate use and final demand.

In [ ]:
u_new = db.u.copy()
Y_new = db.Y.copy()

# pandas >= 3 equivalent of the original groupby(level=[0], axis=1).sum()
U = db.U.copy().loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
Y = Y_new.loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
UY = U + Y

z_new = db.z.copy()

trades_df = {}

for commodity in traded_commodities:
    trades_df[commodity] = pd.DataFrame()
    u_new.loc[:, (slice(None), "Activity", f"{commodity} supply")] *= 0
    oth_activities = [i for i in db.get_index("Activity") if i != f"{commodity} supply"]
    
    for region in db.get_index("Region"):
        u_new.loc[(region, "Commodity", commodity), (region, "Activity", f"{commodity} supply")] = 1

        ee_consumption_u = db.u.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)].sum(0).to_frame().T
        ee_consumption_u.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.u.index.names)

        ee_consumption_Y = db.Y.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))].sum(0).to_frame().T
        ee_consumption_Y.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.Y.index.names)

        u_new.update(ee_consumption_u)
        Y_new.update(ee_consumption_Y)

        u_new.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)] *= 0
        Y_new.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))] *= 0

        trades_df[commodity] = pd.concat([
            trades_df[commodity], 
            UY.loc[:, region] / UY.loc[:, region].sum()
        ], axis=1
        )

z_new.update(u_new)

Update baseline scenario and reset database to coefficients.

In [ ]:
db.update_scenarios(scenario='baseline', z=z_new, Y=Y_new)
db.reset_to_coefficients('baseline')

### Supply-side shock

Use the legacy shock workbook to apply the electricity trade update.

In [ ]:
# db.get_shock_excel("support/trades.xlsx"))
# Electricity Maps trades are proprietary (non-redistributable) -> not in the repo;
# they live in the OneDrive archive under paths['export']/_trades_data.
db.shock_calc(
    os.path.join(paths['export'], "_trades_data", f"trades_{year}.xlsx"),
    z=True, scenario='ee_trades', force_rewrite=True,
)

Export the v2.0 database

In [ ]:
#%% Export the v2.0 database
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v2.0",
        str(year),
        ),
    scenario = 'ee_trades',
    # flows=True,
    # coefficients=True
    )

# Free memory before building v2.1 from a fresh parse.
del db, z, s, u_new, Y_new, z_new, U, Y, UY

---
## v2.1 - new MARIO-native pipeline

Re-parse and re-aggregate the raw table so `v2.1` is independent from the legacy branch. The native electricity update uses EMBER data directly and handles the EXIOBASE Rest-of-World regions through MARIO's packaged region membership.

In [3]:
db3 = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

# Enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.
db3.meta.source = 'EXIOBASE Hybrid 3.3.18'

db3.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/Exiobase Hybrid 3.3.18 with VA/flows in matrix mode (txt/csv).
INFO Parser: Reading flows from txt files.
INFO Parser: Reading files finished.
INFO Parser: Investigating possible identifiable errors.
INFO Parser: parsing database finished.
INFO Parser: state payload ready with 9 canonical blocks.
INFO Parser: txt state ready for SUT.
INFO Metadata: initialized.
WARNING nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure 

In [ ]:
# Supply mixes from nxbase (query API) — see support/nxbase_client.py
from support import nxbase_client as nxc

nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)
print(nxc.get_provenance(nxbase_api, ['EMBER Yearly Electricity Data 2025', 'Electricity Maps']))

# MARIO reads the reduced snapshot from a file; transient, regenerated each run
ember_snapshot_path = 'support/_nxbase_ember_snapshot.csv'
nxc.get_ember_snapshot(nxbase_api).to_csv(ember_snapshot_path, index=False)

db3.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = ember_snapshot_path,
)

nxbase version: 0.1.0 (api: http://127.0.0.1:8000)
source: NXB | Electricity Maps (version None, status active)


Export the v2.1 supply-mix-only intermediate if you want to inspect it in memory. The formal `v2.1` export is written after the trade mix update below.

Pool the trade of the selected commodities. MARIO adds the `" - supply"` / `" - need"` pass-through layer and stores the observed trade shares in the supply block market shares.

In [4]:
traded_commodities = ['Electricity']

# v2-compatible sector names ("Electricity supply"/"Electricity need"):
# the NXS2 namespace rows in nxbase carry exactly these labels.
db3.pool_trade(traded_commodities, supply_suffix=" supply", need_suffix=" need")

INFO Resolver: resolving Z for baseline.
INFO Resolver: trying Z via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved Z via concat.
INFO Resolver: resolving Y for baseline.
INFO Resolver: trying Y via concat.
INFO Resolver: resolved Y via concat.
INFO Resolver: resolving V for baseline.
INFO Resolver: trying V via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved V via concat.
INFO Resolver: resolving E for baseline.
INFO Resolver: trying E via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved E via concat.
INFO pool_trade: pooled trade layer added for ['Electricity'].


### Update trade mixes

Bilateral origin shares come from **nxbase via the query API** (`support/nxbase_client.py`):
one origins-by-destinations matrix per commodity, every destination column summing to 1,
domestic diagonal included. The client logs the nxbase version and source vintages above.
The legacy `trades_{year}.xlsx` workbooks are Electricity Maps proprietary-ToS data
(non-redistributable): they no longer live in the repo — they sit in the OneDrive archive
(`paths['export']/_trades_data/`) and are governed in nxbase (`electricity_maps/`).
`rescale=True` normalizes each destination mix while preserving the destination column totals.

In [ ]:
scenario = 'ee_trades'
if scenario not in db3.scenarios:
    db3.clone_scenario('baseline', scenario)

for commodity in traded_commodities:
    pooled = db3.meta.pooled_trade_map[commodity]
    # v2.1 keeps the proprietary Electricity Maps mix; v3.0 below switches to ENTSO-E.
    trades = nxc.get_trade_matrix(
        nxbase_api, year=year, commodity=commodity,
        source=f"Electricity Maps Import mix {year}",
    )

    db3.update_trade_mix(
        {destination: trades[destination].dropna().to_dict() for destination in trades.columns},
        items = pooled['supply'],
        commodities = pooled['need'],
        scenario = scenario,
        rescale = True,
    )

Export the v2.1 database. The directory is created if it does not exist.

In [ ]:
v21_path = os.path.join(paths['export'], "v2.1", str(year))
os.makedirs(v21_path, exist_ok=True)

db3.to_txt(
    path = v21_path,
    scenario = 'ee_trades',
    )

---
## v3.0 - open ENTSO-E trade mix (will replace v2.1)

Same MARIO-native pipeline as v2.1, but the electricity import mix is the
**open ENTSO-E** physical set from nxbase (source `ENTSO-E electricity import
mix <year>`, physical cross-border flows) instead of the proprietary
Electricity Maps set. ENTSO-E covers the European countries; every other
EXIOBASE region (US, CN, JP, … and the RoW aggregates) is filled
domestic-only. Applied on the same `db3` in a separate `entsoe_trades`
scenario, exported under `v3.0/<year>`.

In [2]:
# v3.0: same MARIO-native pipeline as v2.1, but the electricity import mix is the
# OPEN ENTSO-E physical set from nxbase instead of proprietary Electricity Maps.
scenario3 = 'entsoe_trades'
if scenario3 not in db3.scenarios:
    db3.clone_scenario('baseline', scenario3)

regions = list(db3.get_index('Region'))
for commodity in traded_commodities:
    pooled = db3.meta.pooled_trade_map[commodity]
    trades = nxc.get_trade_matrix(
        nxbase_api, year=year, commodity=commodity,
        source=f"ENTSO-E electricity import mix {year}",
    )
    # ENTSO-E covers the European countries; every other EXIOBASE region
    # (US, CN, JP, ... and the RoW aggregates) is filled domestic-only. So are
    # the origin-only regions ENTSO-E returns as an all-zero destination column
    # (GB, MT): a zero mix would make update_trade_mix reject the destination,
    # so a positive column sum — not mere presence — decides "covered".
    trade_dict = {}
    for dest in regions:
        col = trades[dest] if dest in trades.columns else None
        trade_dict[dest] = (
            col.dropna().to_dict() if (col is not None and col.sum() > 0) else {dest: 1.0}
        )
    db3.update_trade_mix(
        trade_dict,
        items = pooled['supply'],
        commodities = pooled['need'],
        scenario = scenario3,
        rescale = True,
    )

v30_path = os.path.join(paths['export'], "v3.0", str(year))
os.makedirs(v30_path, exist_ok=True)
db3.to_txt(path = v30_path, scenario = 'entsoe_trades')

NameError: name 'db3' is not defined

---
## Footprint comparison: v2.0 legacy vs v2.1 native

Both versions come from this same run. `v2.0` is read back from its export, so the comparison includes the legacy txt round-trip on that side. The old pooled labels are renamed to the new `" - "` convention before alignment.

In [6]:
db_old = mario.parse_from_txt(
    path = os.path.join(paths['export'], "v2.0", str(year), "flows"),
    mode = "flows",
    table = 'SUT',
)

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/nxsut/v2.0/2025/flows in matrix mode (txt/csv).
INFO Parser: Reading flows from txt files.
INFO Parser: Reading files finished.
INFO Parser: Investigating possible identifiable errors.
INFO Parser: parsing database finished.
INFO Parser: state payload ready with 9 canonical blocks.
INFO Parser: txt state ready for SUT.
INFO Metadata: initialized.


In [7]:
gwp = {
    "Carbon dioxide, fossil (air - Emiss)": 1.0,
    "CH4 (air - Emiss)": 25.0,
    "N2O (air - Emiss)": 298.0,
}

def ghg_footprint(f):
    f = f.loc[list(gwp), :].T
    return sum(f[substance] * factor for substance, factor in gwp.items())

f_new = ghg_footprint(db3.query('f', scenarios='ee_trades'))
f_old = ghg_footprint(db_old.f)

# Align the pooled labels of the legacy pipeline to the new naming convention.
item_renames = {}
for commodity in traded_commodities:
    pooled = db3.meta.pooled_trade_map[commodity]
    item_renames[f"{commodity} supply"] = pooled['supply']
    item_renames[f"{commodity} need"] = pooled['need']
f_old = f_old.rename(index=item_renames, level='Item')

comparison = pd.concat([f_old.rename('v2.0'), f_new.rename('v2.1')], axis=1)
comparison['Delta%'] = 100 * (comparison['v2.1'] / comparison['v2.0'] - 1)

comparison['Delta%'].abs().describe()

INFO Resolver: resolving f for ee_trades.
INFO Resolver: trying f via concat.
INFO Resolver: resolved fa via formula build_sut_fa_from_ea_s_u (compute_method=auto, runtime=solve).
INFO Resolver: resolved fc via formula build_sut_fc_from_ea_s_u (compute_method=auto, runtime=solve).
INFO Resolver: resolved f via concat.
INFO Resolver: resolving f for baseline.
INFO Resolver: trying f via concat.
INFO Resolver: resolved waa via formula build_sut_waa_from_s_u (compute_method=auto, runtime=inverse).
INFO Resolver: resolved fa via formula build_sut_fa_from_ea_waa (compute_method=auto, runtime=inverse).
INFO Resolver: resolved wcc via formula build_sut_wcc_from_u_s (compute_method=auto, runtime=inverse).
INFO Resolver: resolved fc via formula build_sut_fc_from_ea_s_wcc (compute_method=auto, runtime=inverse).
INFO Resolver: resolved f via concat.


count    1.416100e+04
mean              inf
std               NaN
min      6.217249e-13
25%      4.530639e-03
50%      1.580826e-02
75%      6.241379e-02
max               inf
Name: Delta%, dtype: float64

In [8]:
# Largest relative deviations across all items.
comparison.loc[comparison['Delta%'].abs().sort_values(ascending=False).index].head(15)

,,,v2.0,v2.1,Delta%
Region,Level,Item,,,
WE,Activity,Other Renewables,0.000000e+00,1.153451e+01,inf
CY,Commodity,Motor Gasoline,0.000000e+00,4.252531e-12,inf
AT,Commodity,Gasoline Type Jet Fuel,0.000000e+00,-1.109496e-12,-inf
GR,Commodity,Gasoline Type Jet Fuel,0.000000e+00,2.441343e-12,inf
HR,Commodity,Coke Oven Coke,0.000000e+00,2.293240e-12,inf
DK,Commodity,Coke Oven Coke,0.000000e+00,7.466614e-13,inf
FI,Commodity,Aviation Gasoline,0.000000e+00,9.898778e-12,inf
IE,Commodity,Kerosene Type Jet Fuel,0.000000e+00,1.884972e-12,inf
FI,Commodity,Biogasoline,0.000000e+00,-7.054724e-12,-inf


In [9]:
# The key check: GHG intensity of the pooled electricity market, per region.
for commodity in traded_commodities:
    need = db3.meta.pooled_trade_map[commodity]['need']
    display(comparison.loc[(slice(None), 'Commodity', need), :].round(4))

,,,v2.0,v2.1,Delta%
Region,Level,Item,,,
AT,Commodity,Electricity - need,77.2954,77.2880,-0.0096
AU,Commodity,Electricity - need,197.9338,197.9343,0.0003
BE,Commodity,Electricity - need,53.8212,53.8200,-0.0021
BG,Commodity,Electricity - need,97.9853,97.9810,-0.0043
BR,Commodity,Electricity - need,31.0747,31.0740,-0.0022
CA,Commodity,Electricity - need,44.4901,44.4912,0.0024
CH,Commodity,Electricity - need,14.3134,12.8124,-10.4868
CN,Commodity,Electricity - need,206.2486,206.2543,0.0028
CY,Commodity,Electricity - need,185.0169,185.0021,-0.0080


In [10]:
# Quick localization of the largest differences by region.
comparison.assign(abs_delta=comparison['Delta%'].abs()).groupby(level='Region')['abs_delta'].max().sort_values(ascending=False).head(10)

Region
AT             inf
PT             inf
FI             inf
IE             inf
DK             inf
GR             inf
CY             inf
WE             inf
HR             inf
EE    3.471436e+05
Name: abs_delta, dtype: float64

---
## Footprint comparison: v2.0 (legacy) vs v3.0 (ENTSO-E) — non-EU focus

v3.0 swaps the proprietary Electricity Maps mix for the open ENTSO-E physical
mix. Non-EU / RoW regions are domestic-only in both pipelines, so their
electricity footprints should be near-identical; any real change concentrates
in the ENTSO-E-covered European countries whose import mix actually differs
between the two datasets.

In [ ]:
# v3.0 (ENTSO-E) vs v2.0 (legacy Electricity Maps). Reuse f_old (v2.0) computed
# above; f from the entsoe_trades scenario is already in the new naming.
f_v30 = ghg_footprint(db3.query('f', scenarios='entsoe_trades'))
comp3 = pd.concat([f_old.rename('v2.0'), f_v30.rename('v3.0')], axis=1)
comp3['Delta%'] = 100 * (comp3['v3.0'] / comp3['v2.0'] - 1)

# Electricity-need GHG intensity per region, split EU (ENTSO-E-covered) vs non-EU.
NON_EU = {'US', 'CN', 'JP', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'ID', 'ZA',
          'CA', 'TW', 'WA', 'WL', 'WE', 'WF', 'WM'}
need = db3.meta.pooled_trade_map['Electricity']['need']
ele = comp3.loc[(slice(None), 'Commodity', need), :].copy()
ele.index = ele.index.get_level_values('Region')
ele['group'] = ['non-EU/RoW' if r in NON_EU else 'EU/ENTSO-E' for r in ele.index]

summary = ele.groupby('group')['Delta%'].agg(
    n='count',
    mean_abs=lambda s: s.abs().mean(),
    max_abs=lambda s: s.abs().max(),
).round(3)
print("Electricity-need footprint, v3.0 vs v2.0 — |Delta%| by group:")
print(summary)
print("\nLargest |Delta%| overall (electricity need):")
display(ele.reindex(ele['Delta%'].abs().sort_values(ascending=False).index).round(3).head(20))